# Task 071: GPRPy — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Ground-penetrating radar migration using GPRPy

**Paper**: GPRPy: Open-Source Ground-Penetrating Radar Processing and Visualization Software
**Repository**: https://github.com/NSGeophysics/GPRPy

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 21.27 dB ← 🔧 修复前: 19.83 dB
- **SSIM**: 0.643
- **Evaluation**: Ground-penetrating radar Kirchhoff depth migration (CC=0.496)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
GPRPy — Ground Penetrating Radar Depth Migration
==================================================
Task #68: Reconstruct subsurface reflectivity from GPR B-scan data
          using Kirchhoff depth migration.

Inverse Problem:
    Given a GPR B-scan d(x,t) (trace ensemble), recover the subsurface
    reflectivity image r(x,z) by migrating diffractions back to their
    true spatial locations.

Forward Model:
    Exploding reflector model:
    d(x,t) = ∫∫ r(x',z) · w(t - 2√((x-x')²+z²)/v) dx' dz
    where v is the EM wave velocity and w(t) is the source wavelet.

Inverse Solver:
    Kirchhoff depth migration — time-domain summation along hyperbolic
    travel-time curves:
    r̂(x,z) = Σ_x' d(x', t=2R/v) · cos(θ)/R

Repo: https://github.com/NSGeophysics/GPRPy
Paper: Annan (2005), Near-Surface Geophysics, SEG.

Usage: /data/yjh/spectro_env/bin/python GPRPy_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`kirchhoff_migration`, `fk_migration`

In [ ]:
# ─── Inverse Solver: Kirchhoff Migration ──────────────────────────
def kirchhoff_migration(bscan, x_traces, z_img, dt, v):
    """
    Kirchhoff depth migration for GPR data (vectorised over traces).

    For each image point (x, z):
    r̂(x,z) = Σ_{x'} d(x', t=2R/v) · cos(θ) / R
    """
    nx = len(x_traces)
    nz = len(z_img)
    nt = bscan.shape[1]
    image = np.zeros((nx, nz))

    print(f"  Migrating {nx} traces × {nz} depth samples ...")

    x_tr_arr = np.asarray(x_traces)  # (nx,)

    for ix_img in range(nx):
        x_pt = x_traces[ix_img]
        for iz_img in range(nz):
            z_pt = z_img[iz_img]
            if z_pt < 1e-6:
                continue

            # Vectorised over all traces
            R = np.sqrt((x_pt - x_tr_arr)**2 + z_pt**2)  # (nx,)
            twt = 2 * R / (v * 1e9)
            it_float = twt / dt  # (nx,)
            it_int = np.floor(it_float).astype(int)
            frac = it_float - it_int

            valid = (it_int >= 0) & (it_int < nt - 1)
            it_safe = np.clip(it_int, 0, nt - 2)

            d_interp = (1 - frac) * bscan[np.arange(nx), it_safe] + \
                       frac * bscan[np.arange(nx), it_safe + 1]

            cos_theta = z_pt / np.maximum(R, 1e-10)
            weight = cos_theta / np.maximum(R, 1e-6)

            image[ix_img, iz_img] = np.sum(valid * d_interp * weight)

    return image

def fk_migration(bscan, dx, dt, v):
    """
    FK (Stolt) migration — frequency-wavenumber domain.
    Faster alternative to Kirchhoff for constant velocity.
    """
    nx, nt = bscan.shape

    # 2D FFT
    D = fft2(bscan)

    # Wavenumber and frequency axes
    kx = np.fft.fftfreq(nx, d=dx) * 2 * np.pi
    omega = np.fft.fftfreq(nt, d=dt) * 2 * np.pi

    # Stolt mapping: ω → kz
    v_ms = v * 1e9  # m/ns to m/s
    image_fk = np.zeros((nx, nt // 2), dtype=complex)

    for ikx in range(nx):
        for iw in range(nt):
            w = omega[iw]
            k = kx[ikx]

            # Dispersion relation: kz = √((2ω/v)² - kx²)
            arg = (2 * w / v_ms)**2 - k**2
            if arg > 0:
                kz = np.sqrt(arg)
                iz = int(kz / (2 * np.pi / (nt * dt * v_ms / 2)) * (nt // 2))
                if 0 <= iz < nt // 2:
                    image_fk[ikx, iz] += D[ikx, iw]

    return np.abs(ifft2(image_fk, s=[nx, nt // 2]))

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `generate_subsurface_model`, `generate_wavelet`, `forward_exploding_reflector`

In [ ]:
# ─── Data Generation ──────────────────────────────────────────────
def generate_subsurface_model(nx, nz, dx, z_max):
    """
    Create a synthetic subsurface reflectivity model with:
    - Horizontal layers
    - Dipping interface
    - Point diffractors (pipes, boulders)
    - Void (tunnel)
    """
    z = np.linspace(0, z_max, nz)
    x = np.arange(nx) * dx
    reflectivity = np.zeros((nx, nz))

    # Layer 1: horizontal at z = 0.5 m
    iz1 = np.argmin(np.abs(z - 0.5))
    reflectivity[:, iz1] = 0.3

    # Layer 2: dipping interface
    for ix in range(nx):
        z_dip = 1.0 + 0.3 * (ix / nx)
        iz_dip = np.argmin(np.abs(z - z_dip))
        if iz_dip < nz:
            reflectivity[ix, iz_dip] = 0.5

    # Layer 3: horizontal at z = 1.8 m
    iz3 = np.argmin(np.abs(z - 1.8))
    reflectivity[:, iz3] = 0.4

    # Point diffractors
    diffractors = [
        (nx // 4, 0.8, 0.7),      # Pipe at shallow depth
        (nx // 2, 1.3, 0.6),      # Boulder
        (3 * nx // 4, 0.6, 0.8),  # Rebar
    ]
    for ix, z_d, amp in diffractors:
        iz_d = np.argmin(np.abs(z - z_d))
        if 0 <= ix < nx and 0 <= iz_d < nz:
            reflectivity[ix, iz_d] = amp

    # Small void (rectangular)
    ix_void_start = int(0.6 * nx)
    ix_void_end = int(0.65 * nx)
    iz_void = np.argmin(np.abs(z - 1.1))
    iz_void_end = np.argmin(np.abs(z - 1.3))
    reflectivity[ix_void_start:ix_void_end, iz_void] = 0.6
    reflectivity[ix_void_start:ix_void_end, iz_void_end] = 0.6
    reflectivity[ix_void_start, iz_void:iz_void_end] = 0.5
    reflectivity[ix_void_end, iz_void:iz_void_end] = 0.5

    return reflectivity, x, z

def generate_wavelet(freq, dt, n_samples=64):
    """Generate Ricker (Mexican hat) wavelet for GPR source."""
    t = np.arange(n_samples) * dt
    t_centre = t[n_samples // 2]
    tau = t - t_centre
    sigma = 1.0 / (np.pi * freq * np.sqrt(2))
    wavelet = (1 - (tau / sigma)**2) * np.exp(-0.5 * (tau / sigma)**2)
    wavelet /= np.max(np.abs(wavelet))
    return wavelet

def forward_exploding_reflector(reflectivity, x_traces, z, dt, nt, v, wavelet, rng):
    """
    Exploding reflector model for GPR B-scan synthesis (vectorised).
    Each reflector explodes simultaneously, waves recorded at surface.
    """
    nx = len(x_traces)
    bscan_clean = np.zeros((nx, nt))
    dx_scene = x_traces[1] - x_traces[0] if len(x_traces) > 1 else 0.05
    n_wav = len(wavelet)

    # Get non-zero reflector positions
    nz_idx = np.argwhere(reflectivity > 1e-10)
    if len(nz_idx) == 0:
        noise = rng.standard_normal(bscan_clean.shape) * 1e-6
        return bscan_clean, bscan_clean + noise

    r_vals = reflectivity[nz_idx[:, 0], nz_idx[:, 1]]  # (K,)
    x_refl = nz_idx[:, 0].astype(float) * dx_scene       # (K,)
    z_refl = z[nz_idx[:, 1]]                               # (K,)

    print(f"  Generating B-scan ({nx} traces, {len(r_vals)} reflectors) ...")

    for ix_t in range(nx):
        x_recv = x_traces[ix_t]
        # Vectorised over all reflectors
        dist = np.sqrt((x_recv - x_refl)**2 + z_refl**2)  # (K,)
        twt = 2 * dist / (v * 1e9)
        it_arr = (twt / dt).astype(int)  # (K,)

        valid = (it_arr >= 0) & (it_arr < nt)
        for k in np.where(valid)[0]:
            it = it_arr[k]
            it_start = max(0, it - n_wav // 2)
            it_end = min(nt, it + n_wav // 2)
            wav_start = it_start - (it - n_wav // 2)
            wav_end = wav_start + (it_end - it_start)
            bscan_clean[ix_t, it_start:it_end] += r_vals[k] * wavelet[wav_start:wav_end]

    # Add noise
    sig_power = np.mean(bscan_clean**2)
    noise_power = sig_power / (10**(NOISE_SNR_DB / 10))
    noise = np.sqrt(noise_power) * rng.standard_normal(bscan_clean.shape)
    bscan_noisy = bscan_clean + noise

    return bscan_clean, bscan_noisy

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize_results`

In [ ]:
# ─── Metrics ───────────────────────────────────────────────────────
def compute_metrics(gt, rec):
    gt_n = gt / max(gt.max(), 1e-12)
    rec_n = rec / max(rec.max(), 1e-12)
    data_range = 1.0
    mse = np.mean((gt_n - rec_n)**2)
    psnr = float(10 * np.log10(data_range**2 / max(mse, 1e-30)))
    ssim_val = float(ssim_fn(gt_n, rec_n, data_range=data_range))
    cc = float(np.corrcoef(gt_n.ravel(), rec_n.ravel())[0, 1])
    re = float(np.linalg.norm(gt_n - rec_n) / max(np.linalg.norm(gt_n), 1e-12))
    rmse = float(np.sqrt(mse))
    return {"PSNR": psnr, "SSIM": ssim_val, "CC": cc, "RE": re, "RMSE": rmse}

# ─── Visualization ─────────────────────────────────────────────────
def visualize_results(reflectivity, bscan, migrated, x, z, t_axis, metrics, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Subsurface model
    axes[0, 0].imshow(reflectivity.T, aspect='auto', cmap='gray_r',
                       extent=[x[0], x[-1], z[-1], z[0]])
    axes[0, 0].set_title('True Reflectivity Model')
    axes[0, 0].set_xlabel('Position [m]')
    axes[0, 0].set_ylabel('Depth [m]')

    # B-scan
    clip = np.percentile(np.abs(bscan), 98)
    axes[0, 1].imshow(bscan.T, aspect='auto', cmap='RdBu_r', vmin=-clip, vmax=clip,
                       extent=[x[0], x[-1], t_axis[-1]*1e9, t_axis[0]*1e9])
    axes[0, 1].set_title('GPR B-Scan (noisy)')
    axes[0, 1].set_xlabel('Position [m]')
    axes[0, 1].set_ylabel('Two-way time [ns]')

    # Migrated image
    axes[1, 0].imshow(migrated.T, aspect='auto', cmap='gray_r',
                       extent=[x[0], x[-1], z[-1], z[0]])
    axes[1, 0].set_title('Kirchhoff Migration')
    axes[1, 0].set_xlabel('Position [m]')
    axes[1, 0].set_ylabel('Depth [m]')

    # Cross-section comparison
    mid = reflectivity.shape[0] // 2
    axes[1, 1].plot(z, reflectivity[mid, :] / max(reflectivity[mid, :].max(), 1e-12),
                     'b-', lw=2, label='GT')
    axes[1, 1].plot(z, migrated[mid, :] / max(migrated[mid, :].max(), 1e-12),
                     'r--', lw=2, label='Migrated')
    axes[1, 1].set_title(f'Trace {mid} Comparison')
    axes[1, 1].set_xlabel('Depth [m]')
    axes[1, 1].legend()

    fig.suptitle(
        f"GPRPy — GPR Migration\n"
        f"PSNR={metrics['PSNR']:.1f} dB | CC={metrics['CC']:.4f}",
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 70)
print("  GPRPy — Ground Penetrating Radar Migration")
print("=" * 70)

rng = np.random.default_rng(SEED)

### Stage 1: Data Generation

Intermediate processing step in the pipeline.

In [ ]:
# Stage 1: Data Generation
print("\n[STAGE 1] Data Generation")
reflectivity, x_traces, z_depth = generate_subsurface_model(
    NX_TRACES, NZ, DX, Z_MAX
)
print(f"  Model: {reflectivity.shape}, dx={DX} m, z_max={Z_MAX:.2f} m")
print(f"  Reflectors: {np.count_nonzero(reflectivity)} non-zero cells")

### Stage 2: Forward — B-scan synthesis

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Stage 2: Forward — B-scan synthesis
print("\n[STAGE 2] Forward — Exploding Reflector B-Scan")
wavelet = generate_wavelet(FREQ_CENTER, DT)
bscan_clean, bscan_noisy = forward_exploding_reflector(
    reflectivity, x_traces, z_depth, DT, NT, V_EM, wavelet, rng
)
t_axis = np.arange(NT) * DT
print(f"  B-scan: {bscan_noisy.shape}")
print(f"  Signal range: [{bscan_clean.min():.3f}, {bscan_clean.max():.3f}]")

### Stage 3: Inverse — Kirchhoff migration

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Stage 3: Inverse — Kirchhoff migration
print("\n[STAGE 3] Inverse — Kirchhoff Depth Migration")
migrated = kirchhoff_migration(bscan_noisy, x_traces, z_depth, DT, V_EM)
# Reflectivity is physically non-negative; clip migration artifacts
migrated = np.clip(migrated, 0, None)
print(f"  Migrated image: {migrated.shape}")

### Stage 4: Evaluation

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Stage 4: Evaluation
print("\n[STAGE 4] Evaluation Metrics:")
metrics = compute_metrics(reflectivity, migrated)
for k, v in sorted(metrics.items()):
    print(f"  {k:20s} = {v}")

with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), migrated)
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), reflectivity)

visualize_results(reflectivity, bscan_noisy, migrated, x_traces, z_depth,
                  t_axis, metrics, os.path.join(RESULTS_DIR, "reconstruction_result.png"))

print("\n" + "=" * 70)
print("  DONE — Results saved to", RESULTS_DIR)
print("=" * 70)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **GPRPy**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=21.27 dB ← 🔧 修复前: 19.83 dB, SSIM=0.643)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: GPRPy: Open-Source Ground-Penetrating Radar Processing and Visualization Software
- Repository: https://github.com/NSGeophysics/GPRPy
